# Base-rate merged results

Explore `data/base_rate/base_rate_merged_results.csv` from a benchmark run.

Scores use `score_outcome` (`normative`, `biased`, `off_target`, `unparseable`). The **normative score** is 1 when `score_outcome == "normative"`, else 0.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

MERGED_CSV = ROOT / "data" / "base_rate" / "base_rate_merged_results.csv"
if not MERGED_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {MERGED_CSV}. Run the base-rate benchmark first "
        "(benchmark/base-rate-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)
df["normative_score"] = (df["score_outcome"] == "normative").astype(int)

print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Rows: 10
Models: ['google/gemini-3-flash-preview']
Vignettes: 2


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score_outcome,normative.1,biased,lure_matched,normative_score
0,actor_waiter_overlap__overlap__mc_full_no_probs,actor waiter overlap,overlap,small,mc_full,False,mc_full_no_probs,You are a statistical consultant. Your task is...,False,False,...,NaN,E,4,mc_full,True,biased,False,True,partition shortcut (assumes P(C∩D|A)=0),0
1,actor_waiter_overlap__overlap__mc_full_probs,actor waiter overlap,overlap,small,mc_full,True,mc_full_probs,You are a statistical consultant. Your task is...,False,False,...,NaN,D,4,mc_full,True,biased,False,True,partition shortcut (assumes P(C∩D|A)=0),0
2,actor_waiter_overlap__overlap__mc_numeric_no_p...,actor waiter overlap,overlap,small,mc_numeric,False,mc_numeric_no_probs,You are a statistical consultant. Your task is...,False,False,...,NaN,E,4,mc_numeric,True,biased,False,True,partition shortcut (assumes P(C∩D|A)=0),0
3,actor_waiter_overlap__overlap__mc_numeric_probs,actor waiter overlap,overlap,small,mc_numeric,True,mc_numeric_probs,You are a statistical consultant. Your task is...,False,False,...,NaN,D,5,mc_numeric,True,biased,False,True,q_d*s_d only (second pathway; assumes partition),0
4,actor_waiter_overlap__overlap__open_no_probs,actor waiter overlap,overlap,small,open,False,open_no_probs,You are a statistical consultant. Your task is...,False,True,...,0.1,NaN,2,open,True,normative,True,False,NaN,1


## Scores by `response_type`

In [2]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]
SCORE_OUTCOME_ORDER = ["normative", "biased", "off_target", "unparseable"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, outcome mix, and mean normative score for each group value."""
    counts = pd.crosstab(df[group_col], df["score_outcome"])
    for outcome in SCORE_OUTCOME_ORDER:
        if outcome not in counts.columns:
            counts[outcome] = 0
    counts = counts[SCORE_OUTCOME_ORDER]

    summary = counts.copy()
    summary["n"] = summary.sum(axis=1)
    summary["normative_rate"] = df.groupby(group_col)["normative_score"].mean()
    summary["normative_pct"] = (summary["normative_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

score_outcome,normative,biased,off_target,unparseable,n,normative_rate,normative_pct
response_type,,,,,,,
open,1,1,0,0,2,0.5,50.0
mc_numeric,0,4,0,0,4,0.0,0.0
mc_full,0,4,0,0,4,0.0,0.0


## Scores by `variant`

In [3]:
VARIANT_ORDER = [
    "open_probs",
    "open_no_probs",
    "mc_numeric_probs",
    "mc_numeric_no_probs",
    "mc_full_probs",
    "mc_full_no_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

score_outcome,normative,biased,off_target,unparseable,n,normative_rate,normative_pct
variant,,,,,,,
open_probs,0,1,0,0,1,0.0,0.0
open_no_probs,1,0,0,0,1,1.0,100.0
mc_numeric_probs,0,2,0,0,2,0.0,0.0
mc_numeric_no_probs,0,2,0,0,2,0.0,0.0
mc_full_probs,0,2,0,0,2,0.0,0.0
mc_full_no_probs,0,2,0,0,2,0.0,0.0


## Scores by `vignette_name`

In [ ]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

## Optional: split by model when multiple LLMs are present

In [4]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["normative_score"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["normative_score"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

Single model in file — see tables above.
